# React Agent from scratch in Python

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [2]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
llm.invoke("Hi").content

"It's nice to meet you. Is there something I can help you with or would you like to chat?"

In [3]:
message=[{"role":"system","content":"you are a helpful assistant"},
         {"role":"user","content":"hi how are you?"}]

result = llm.invoke(message)
print(result.content)

I'm doing well, thank you for asking. I'm a helpful assistant, here to provide information and assist with any questions or tasks you may have. How can I help you today?


In [4]:
class Chatbot:
    def __init__(self,system=""):
        self.system=system
        self.message=[]
        if self.system:
            self.message.append({"role":"system","content":system})

    def __call__(self,message):
        self.message.append({"role":"user","content":message})
        result=self.execute()
        self.message.append({"role":"assistant","content":result})
        return result
        
    def execute(self):
        llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)
        result = llm.invoke(self.message)
        return result.content

In [5]:
bot = Chatbot(system = "You are a helpful assistant.")

In [6]:
bot("Hi how are you")

"I'm doing well, thank you for asking. I'm here to assist you with any questions or topics you'd like to discuss. How about you? How's your day going so far?"

In [7]:
bot.message

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'Hi how are you'},
 {'role': 'assistant',
  'content': "I'm doing well, thank you for asking. I'm here to assist you with any questions or topics you'd like to discuss. How about you? How's your day going so far?"}]

In [8]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop your output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.


Your available actions are:
calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point
syntax if necessary

wikipedia:
e.g. wikipedia: Django
Returns a summary from searching Wikipedia

simon_blog_search:
e.g. simon_blog_search: Python
Search Simon's blog for that term

Example session:
Question: What is the capital of France?
Thought: I should look up France on Wikipedia
Action: wikipedia: France
PAUSE

You will be called again with this:
Observation: France is a country. The capital is Paris.

You then output:
Answer: The capital of France is Paris

Please Note: if you get basic conversation questions like "hi","hello","how are you?",\n
you have to answer "hi","hello","i am good".
""".strip()

Understand the Regular expression
Pattern Breakdown:

^: This matches the start of a string. It means the string must begin with what follows.
Action:: This is a literal match. It means the string must have the text "Action:" at the beginning.
(\w+):
1. The parentheses () define a capture group. This allows you to extract part of the string that matches this section.
2. \w+ matches one or more word characters (letters, digits, and underscores). This will capture a word that follows "Action:".
:: This is a literal colon that separates the word matched by (\w+) from the rest of the string.
(.*):
1. This is another capture group, where .* matches any character (.) zero or more times (*), which means it captures everything that comes after the second colon.
What does it do?
This regex is looking for a string that:

1. Starts with the text "Action:".
2. Has a word right after it, separated by a colon.
3. Then, after another colon, it captures everything that follows.

In [28]:
import re
action_re = re.compile(r"Action: (\w+): (.*)")

In [10]:
import httpx
def wikipedia(query):
    response = httpx.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    })
    return response.json()["query"]["search"][0]["snippet"]

In [11]:
import httpx
def simon_blog_search(query):
    response = httpx.get("https://datasette.simonwillison.net/simonwillisonblog.json", params={
        "sql": """
        select
          blog_entry.title || ': ' || substr(html_strip_tags(blog_entry.body), 0, 1000) as text,
          blog_entry.created
        from
          blog_entry join blog_entry_fts on blog_entry.rowid = blog_entry_fts.rowid
        where
          blog_entry_fts match escape_fts(:q)
        order by
          blog_entry_fts.rank
        limit
          1
        """.strip(),
        "_shape": "array",
        "q": query,
    })
    return response.json()[0]["text"]

In [12]:
def calculate(number):
    return eval(number)

In [13]:
calculate("2+2")

4

In [14]:
calculate("4 * 7 / 3")

9.333333333333334

In [15]:
known_actions = {
    "wikipedia": wikipedia,
    "calculate": calculate,
    "simon_blog_search": simon_blog_search
}

In [18]:
question="hi how are you?"

In [19]:
bot = Chatbot(prompt)

In [20]:
bot

In [21]:
question="hello again?"

In [22]:
next_prompt = question

In [23]:
result = bot(next_prompt)
print(result)

Thought: I've been asked a greeting, I should return a greeting

Action: hello

PAUSE



In [24]:
actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]

In [25]:
actions

[<re.Match object; span=(0, 13), match='Action: hello'>]

In [26]:
def query(question, max_turns=5):
    i = 0
    bot = Chatbot(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
        if actions:
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception(f"Unknown action: {action}: {action_input}")
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = f"Observation: {observation}"
        else:
            return result

In [29]:
query("hi")

Thought: The person has said hi, so I should respond with a greeting.

Action: hello
PAUSE

(I don't have any observations to make in this case!)

Answer: hello


"Thought: The person has said hi, so I should respond with a greeting.\n\nAction: hello\nPAUSE\n\n(I don't have any observations to make in this case!)\n\nAnswer: hello"

In [30]:
query("hello")

Thought: The user has greeted me, I should respond in kind.

Action: hello

PAUSE


'Thought: The user has greeted me, I should respond in kind.\n\nAction: hello\n\nPAUSE'

In [31]:
query("What are you doing?")

Thought: I'm running in a loop where I process a question, perform actions based on the question, and then output an answer.

Action: hello
PAUSE


"Thought: I'm running in a loop where I process a question, perform actions based on the question, and then output an answer.\n\nAction: hello\nPAUSE"

In [32]:
query("Has Simon written about AI?")

Thought: I should see if Simon has written about AI in his blog.

Action: simon_blog_search: AI
PAUSE
 -- running simon_blog_search AI
Observation: Talking AI and jobs with Natasha Zouves for News Nation: I was interviewed by News Nation's Natasha Zouves about the very complicated topic of how we should think about AI in terms of threatening our jobs and careers. I previously talked with Natasha two years ago about Microsoft Bing.

I'll be honest: I was nervous about this one. I'm not an economist and I didn't feel confident talking about this topic!

I do find the challenge of making recent advances in AI and LLMs accessible to a general audience absolutely fascinating though, so I took the risk and agreed to the interview.

I think it came out very well. The full hour long video is now available on the News Nation YouTube channel, or as an audio podcast on iTunes or on Spotify.

 

I made my own transcript of the video (using MacWhisper) and fed it into the new Claude Opus 4 model to

'Thought: So, it seems Simon has written about AI, specifically about an interview he did with Natasha Zouves on News Nation. He discusses the challenges of making AI accessible to a general audience and shares a transcript and outline of the episode generated by the Claude Opus 4 model.\n\nAnswer: Yes, Simon has written about AI.'

In [33]:
print(query("Fifteen * twenty five"))

Thought: I need to run a calculation using Python's syntax to get the result of 15 * 25 / 1

Action: calculate: 15 * 25 / 1
PAUSE
 -- running calculate 15 * 25 / 1
Observation: 375.0
Answer: 375.0
Answer: 375.0
